### Temporary code to rescuing 520 removed genes across libraries

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
def process_discarded_sgrna(discarded_df):
    discarded = list(set(
        discarded_df["multi_target_guides"].dropna().tolist() +
        discarded_df["single_mismatch_guides"].dropna().tolist() +
        discarded_df["pam_distal_double_mismatch_guides"].dropna().tolist()
    ))
    return discarded

def get_removed_genes_all_libraries(filepath):
    removed_genes = pd.read_csv(filepath)

    removed_gene_list_combined = {
        "Avana":    removed_genes["avana_removed_genes_all"].dropna().tolist(),
        "Brunello": removed_genes["brunello_removed_genes_all"].dropna().tolist(),
        "TKOv3":    removed_genes["toronto_removed_genes_all"].dropna().tolist(),
        "Yusa":     removed_genes["yusa_removed_genes_all"].dropna().tolist(),
        "Jacquere": removed_genes["jacquere_removed_genes_all"].dropna().tolist(),
    }

    return list(set.intersection(*[set(v) for v in removed_gene_list_combined.values()]))

In [12]:
# Import list discarded sgRNA

brunello = pd.read_csv("../data/guiderefine_output/T2T-CHM13/broadgpp-brunello-library-contents_disposed_sgRNAs.tsv", sep = "\t")
tkov3 = pd.read_csv("../data/guiderefine_output/T2T-CHM13/tkov3_guide_sequence_disposed_sgRNAs.tsv", sep = "\t")
yusa = pd.read_csv("../data/guiderefine_output/T2T-CHM13/yusa_hcrispr_ko_grnas_disposed_sgRNAs.tsv", sep = "\t")
avana = pd.read_csv("../data/guiderefine_output/T2T-CHM13/avana_library_disposed_sgRNAs.tsv", sep = "\t")
jacquere = pd.read_csv("../data/guiderefine_output/T2T-CHM13/Jacquere_PerGuideAnnotations_Quota4_disposed_sgRNAs.tsv", sep = "\t")

# Import list removed genes all libraries
# path below is to be filled
# this should be output 520 protein-coding genes unrepresented across all 5 CRISPR-KO libraries
removed_genes_all_libraries = get_removed_genes_all_libraries("../data/removed_genes_survey/removed_genes_all_library.csv")


In [13]:
# Import original library files (no header, columns: guide_id, sequence, gene)
_lib_cols = ["guide_id", "sequence", "gene"]

avana_lib    = pd.read_csv("../data/library_data/original_library/avana_library.tsv", sep="\t", header=None, names=_lib_cols)
brunello_lib = pd.read_csv("../data/library_data/original_library/broadgpp-brunello-library-contents.tsv", sep="\t", header=None, names=_lib_cols)
tkov3_lib    = pd.read_csv("../data/library_data/original_library/tkov3_guide_sequence.tsv", sep="\t", header=None, names=_lib_cols)
yusa_lib     = pd.read_csv("../data/library_data/original_library/yusa_hcrispr_ko_grnas.tsv", sep="\t", header=None, names=_lib_cols)
jacquere_lib = pd.read_csv("../data/library_data/original_library/Jacquere_PerGuideAnnotations_Quota4.tsv", sep="\t", header=None, names=_lib_cols)

In [ ]:
def get_surviving_guides_from_report(report_path, removed_genes):
    """
    Read 'counted total sgRNA' from the Genes Report sheet of a full_report.xlsx.
    This column accounts for all disposal reasons: off-target, mismatch, intronic,
    not-targeting-anywhere, and guide corrections — unlike the disposed_sgRNAs TSV
    which only contains guide IDs for 3 of those categories.
    """
    df = pd.read_excel(report_path, sheet_name="Genes Report")
    counts = df.set_index("gene")["counted total sgRNA"]
    return counts.reindex(removed_genes, fill_value=0)


def count_surviving_guides_all_libraries(report_paths, removed_genes):
    """
    Returns a DataFrame (genes × libraries) of surviving guide counts,
    read directly from each library's full_report.xlsx.
    """
    results = {}
    for lib_name, path in report_paths.items():
        results[lib_name] = get_surviving_guides_from_report(path, removed_genes)
    return pd.DataFrame(results)

In [ ]:
report_paths = {
    "Avana":    "../data/guiderefine_output/T2T-CHM13/avana_library_full_report.xlsx",
    "Brunello": "../data/guiderefine_output/T2T-CHM13/broadgpp-brunello-library-contents_full_report.xlsx",
    "TKOv3":    "../data/guiderefine_output/T2T-CHM13/tkov3_guide_sequence_full_report.xlsx",
    "Yusa":     "../data/guiderefine_output/T2T-CHM13/yusa_hcrispr_ko_grnas_full_report.xlsx",
    "Jacquere": "../data/guiderefine_output/T2T-CHM13/Jacquere_PerGuideAnnotations_Quota4_full_report.xlsx",
}

surviving_guides = count_surviving_guides_all_libraries(report_paths, removed_genes_all_libraries)

print(f"Shape: {surviving_guides.shape}  — expected (520, 5)")
surviving_guides

In [24]:
surviving_guides["sum_guides"] = surviving_guides.sum(axis=1)
surviving_guides

,Avana,Brunello,TKOv3,Yusa,Jacquere,sum_guides
gene,,,,,,
HSPA6,0,1,0,1,1,3
PNRC2,0,0,0,0,0,0
RBMS2,0,2,1,0,2,5
VPS35,1,1,1,2,2,7
TBCA,0,0,1,0,1,2
...,...,...,...,...,...,...
PRAMEF10,0,0,0,0,1,1
DIPK1A,0,0,0,0,4,4
STX18,2,2,2,1,2,9


In [ ]:
surviving_guides[surviving_guides["sum_guides"] > 2].sort_values(by="sum_guides", ascending=False)

,Avana,Brunello,TKOv3,Yusa,Jacquere,sum_guides
gene,,,,,,
SLAIN1,4,4,4,7,4,23
SCRN3,4,4,4,5,4,21
PGPEP1L,4,4,4,5,4,21
DMPK,4,4,4,5,4,21
SYT3,4,4,4,5,4,21
...,...,...,...,...,...,...
NCF1,0,1,0,1,1,3
PSG4,0,1,1,0,1,3
NHP2,0,1,0,1,1,3


In [26]:
surviving_guides.to_excel("surviving_guides.xlsx")